In [18]:
import rasterio
import numpy as np
import pandas as pd
from rasterio.windows import Window
import matplotlib.pyplot as plt

In [33]:
# Load Satellite Image and Reference Image
reference_tif = "2019_30m_cdls_cropped.tif"
satelite_tif = "2020_30m_cdls_cropped.tif" #switch this oue to sentinel-2 image later

In [34]:
#print all grayscale values
with rasterio.open(reference_tif) as ref_src:
    ref_image = ref_src.read(1)  # Read as a single-band (grayscale) array

# Get unique grayscale values
unique_values = np.unique(ref_image)

# Print the values
print("Unique grayscale values in the reference image:")
print(unique_values)

Unique grayscale values in the reference image:
[  0   1   4   5   6  12  13  14  21  24  25  27  28  30  36  37  41  42
  43  44  49  50  53  54  56  57  58  59  61  66  68  70 111 121 122 123
 124 131 141 142 143 152 176 190 195 205 206 219 222 225 229 243 246]


In [36]:
# Define crop mapping (grayscale values to crop labels)
CROP_MAPPING = {
    1: "Corn",
    4: "Soybean",
    5: "Wheat",
    6: "Alfalfa",
    12: "Sugar Beet",
    13: "Other"
}

In [37]:
# Define the grid size
GRID_SIZE = 1  # Adjust based on resolution needs

In [38]:
with rasterio.open(satellite_tif) as sat_src, rasterio.open(reference_tif) as ref_src:
    width, height = sat_src.width, sat_src.height
    grid_data = []  # Store labeled data
    
    # Loop over grid cells
    for y in range(0, height, GRID_SIZE):
        for x in range(0, width, GRID_SIZE):
            # Define window for the grid section
            window = Window(x, y, GRID_SIZE, GRID_SIZE)
            
            # Read the corresponding section from the reference image
            ref_patch = ref_src.read(1, window=window)
            
            # Find the most common grayscale value in the patch
            unique, counts = np.unique(ref_patch, return_counts=True)
            dominant_grayscale = unique[np.argmax(counts)]
            
            # Map grayscale value to crop type
            crop_type = CROP_MAPPING.get(dominant_grayscale, "Unknown")
            
            # Store grid location and crop type
            grid_data.append({
                "x": x,
                "y": y,
                "crop_type": crop_type
            })

In [39]:
# Convert to DataFrame and Save
df = pd.DataFrame(grid_data)
df.to_csv("labeled_grid.csv", index=False)